In [1]:
import os
import json
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# Constants
PERSONAL_DB = "/scratch/gpfs/GRIFFITHS/hl4291/personal.db"
EPSILON = 1e-6
LIMIT_N = 1000  # Use None for full dataset

In [ ]:
# 1. Connect
# src_dir = os.path.dirname(os.path.abspath(__file__))
print(f"Connecting to {PERSONAL_DB}...")
conn = duckdb.connect(database=PERSONAL_DB, read_only=False)

# 2. SQL Pre-calculation
limit_clause = f"LIMIT {LIMIT_N}" if LIMIT_N is not None else ""
print(f"Creating SQL tables _selected_moves and stats (limit={LIMIT_N or 'FULL'})...")

conn.execute(f"""
    CREATE OR REPLACE TABLE _selected_moves AS
    SELECT 
        gid,
        move_ply,
        player_clock_time,
        opponent_clock_time, 
        log(player_clock_time + {EPSILON}) as ln_player_clock_time, 
        log(opponent_clock_time + {EPSILON}) as ln_opponent_clock_time, 
        log(move_time + {EPSILON}) as ln_move_time,
        ntile(10) over (order by move_ply) as qbin_move_ply
    FROM (
        SELECT *
        FROM selected_moves
        {limit_clause}
    );
""")

# 3. Get Counts from processed table
print("Getting processed dataset counts...")
n_games = conn.execute("SELECT count(distinct gid) FROM _selected_moves").fetchone()[0]
n_moves = conn.execute("SELECT count(*) FROM _selected_moves").fetchone()[0]
print(f"Total Games: {n_games:,} | Total Moves: {n_moves:,}")

Connecting to /scratch/gpfs/GRIFFITHS/hl4291/personal.db...
Creating SQL tables _selected_moves and stats (limit=1000)...
Getting processed dataset counts...
Total Games: 15 | Total Moves: 1,000


In [10]:
df = conn.execute("SELECT * FROM _selected_moves").df()

In [11]:
df

,gid,move_ply,player_clock_time,opponent_clock_time,ln_player_clock_time,ln_opponent_clock_time,ln_move_time,move_ply_qbin
0,202311001003581,1,600,600,2.778151,2.778151,NaN,1
1,202311001022662,1,600,600,2.778151,2.778151,NaN,1
2,202311001034120,1,600,600,2.778151,2.778151,NaN,1
3,202311001063291,1,600,600,2.778151,2.778151,NaN,1
4,202311001094579,1,600,600,2.778151,2.778151,NaN,1
...,...,...,...,...,...,...,...,...
995,202311001145048,150,15,3,1.176091,0.477121,6.020601e-01,10
996,202311001145048,151,3,11,0.477121,1.041393,4.342943e-07,10
997,202311001145048,152,11,2,1.041393,0.301030,6.020601e-01,10
998,202311001145048,153,2,7,0.301030,0.845098,-6.000000e+00,10
